# Phase 5: Targeted Analysis 4: τ Threshold Robustness

## Overview

The pareto sweep selects τ* for each model by balancing circuit size and faithfulness on the control band.
A natural concern is whether the transfer efficiency result is an artifact of this specific τ* choice.
This notebook answers that question by re-evaluating control-band circuits at τ_low and τ_high
(immediate neighbors of τ* in the 11-point log-uniform sweep) across all 5 test bands.

## Key Question

**Does transfer efficiency remain stable as τ varies over a 2-step range around τ*?**
- If yes (CV < 5%), τ* selection does not drive the result.
- Transfer efficiency is defined as: mean(circuit_acc on {low, med, high, very_high}) / circuit_acc on control

## Hypothesis

- **H-R.D2.1**: Transfer efficiency is stable (CV < 5%) across τ_low/τ*/τ_high for all 5 models.
- **H-R.D2.2**: Circuit size (n_edges) varies monotonically with τ, confirming expected behavior.

## Constraint

Only control-band circuits exist at alternative τ values (per-band circuits at τ!=τ* would require
full ACDC reruns). This is the correct test: τ* is selected on control data, so control-band
robustness is the relevant question.

## Notebook Structure

1. Setup & Data Loading
2. Transfer Efficiency Computation
3. Results Table (paper-ready)
4. Visualization
5. Summary

## Data Sources

- '05_Phase_Targeted/outputs/threshold_robustness/eval_results.csv'
  - 75 rows: 5 models x 3 τ variants (low/star/high) x 5 test bands
  - Generated by 'lsc_threshold_robustness_eval.py'

In [1]:
import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Paths
ANALYSIS_ROOT = Path("LSC_circuit_analysis")
PHASE5_DIR = ANALYSIS_ROOT / "05_Phase_Targeted"
INPUT_DIR = PHASE5_DIR / "outputs" / "threshold_robustness"
ANALYSIS_DIR = INPUT_DIR / "analysis"
VIZ_DIR = INPUT_DIR / "viz"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
VIZ_DIR.mkdir(parents=True, exist_ok=True)

# Constants
MODELS = ["pythia-70m", "pythia-160m", "pythia-410m", "pythia-1b", "pythia-1.4b"]
BANDS = ["low", "medium", "high", "very_high", "control"]
TAU_ORDER = ["low", "star", "high"]  # ascending τ value (low=more edges, high=fewer)
TAU_LABELS = {"low": "τ_low", "star": "τ*", "high": "τ_high"}

MODEL_COLORS = {
    "pythia-70m": "#2196F3",
    "pythia-160m": "#4CAF50",
    "pythia-410m": "#FF9800",
    "pythia-1b": "#F44336",
    "pythia-1.4b": "#9C27B0",
}
TAU_COLORS = {"low": "#1565C0", "star": "#2E7D32", "high": "#BF360C"}
TAU_MARKERS = {"low": "o", "star": "s", "high": "^"}

sns.set_theme(style="whitegrid", font_scale=1.1)


def save_figure(fig, filename):
    path = VIZ_DIR / filename
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")


# Load evaluation results
df = pd.read_csv(INPUT_DIR / "eval_results.csv")
print(f"Loaded {len(df)} rows from eval_results.csv")
print(f"Columns: {list(df.columns)}")
print(df.head(10).to_string())

Loaded 75 rows from eval_results.csv
Columns: ['model', 'tau_variant', 'tau_value', 'n_edges', 'total_edges', 'size_fraction', 'test_band', 'circuit_accuracy', 'circuit_kl_div', 'base_accuracy']
        model tau_variant  tau_value  n_edges  total_edges  size_fraction  test_band  circuit_accuracy  circuit_kl_div  base_accuracy
0  pythia-70m         low   0.000631      590         1324       0.445619        low          0.271111        0.101982       0.284444
1  pythia-70m         low   0.000631      590         1324       0.445619     medium          0.386667        0.118053       0.422222
2  pythia-70m         low   0.000631      590         1324       0.445619       high          0.435556        0.131454       0.457778
3  pythia-70m         low   0.000631      590         1324       0.445619  very_high          0.604444        0.118557       0.653333
4  pythia-70m         low   0.000631      590         1324       0.445619    control          0.551111        0.144698       0.595556
5

## 2. Transfer Efficiency Computation

For control-band circuits at each τ variant:

'''
transfer_eff(model, τ) = mean(circuit_acc on {low, med, high, very_high}) / circuit_acc on control
'''

This ratio measures how well a control-band circuit transfers to frequency-stratified test bands,
relative to its within-distribution (control) performance.

In [2]:
# Separate same-band (control) and cross-band accuracy
same = df[df["test_band"] == "control"][
    [
        "model",
        "tau_variant",
        "tau_value",
        "n_edges",
        "total_edges",
        "size_fraction",
        "circuit_accuracy",
        "base_accuracy",
    ]
].copy()
same = same.rename(
    columns={"circuit_accuracy": "same_acc", "base_accuracy": "base_acc_control"}
)

cross = (
    df[df["test_band"] != "control"]
    .groupby(["model", "tau_variant"])["circuit_accuracy"]
    .mean()
    .reset_index()
    .rename(columns={"circuit_accuracy": "cross_acc"})
)

te = same.merge(cross, on=["model", "tau_variant"])
te["transfer_eff"] = te["cross_acc"] / te["same_acc"]
te["tau_variant"] = pd.Categorical(
    te["tau_variant"], categories=TAU_ORDER, ordered=True
)
te = te.sort_values(["model", "tau_variant"]).reset_index(drop=True)

print("Transfer efficiency per (model, τ variant):")
print(
    te[
        [
            "model",
            "tau_variant",
            "tau_value",
            "n_edges",
            "size_fraction",
            "same_acc",
            "cross_acc",
            "transfer_eff",
        ]
    ].to_string(index=False)
)

Transfer efficiency per (model, τ variant):
      model tau_variant  tau_value  n_edges  size_fraction  same_acc  cross_acc  transfer_eff
pythia-1.4b         low   0.000251     3770       0.046785  0.942222   0.860000      0.912736
pythia-1.4b        star   0.000631     2097       0.026024  0.920000   0.860000      0.934783
pythia-1.4b        high   0.001580     1197       0.014855  0.831111   0.731111      0.879679
pythia-160m         low   0.000251     2138       0.186448  0.920000   0.916667      0.996377
pythia-160m        star   0.000631     1396       0.121741  0.924444   0.917778      0.992788
pythia-160m        high   0.001580      904       0.078835  0.875556   0.844444      0.964467
  pythia-1b         low   0.000631     1425       0.142372  0.973333   0.941111      0.966895
  pythia-1b        star   0.001580      939       0.093816  0.937778   0.894444      0.953791
  pythia-1b        high   0.003980      551       0.055050  0.822222   0.755556      0.918919
pythia-410m     

## 3. Results Table

Paper-ready table: 5 models x 3 τ variants. Columns: τ value, n_edges, size%, TE.
Last column: CV% = coefficient of variation across the 3 τ variants per model.

In [3]:
# Build paper table
rows = []
for model in MODELS:
    msub = te[te["model"] == model].sort_values("tau_variant")

    te_values = msub["transfer_eff"].values
    cv_pct = (
        (np.std(te_values) / np.mean(te_values) * 100)
        if np.mean(te_values) > 0
        else float("nan")
    )

    row = {"model": model}
    for _, r in msub.iterrows():
        v = r["tau_variant"]
        row[f"{v}_tau"] = f"{r['tau_value']:.2e}"
        row[f"{v}_edges"] = int(r["n_edges"])
        row[f"{v}_size"] = f"{r['size_fraction']:.1%}"
        row[f"{v}_te"] = f"{r['transfer_eff']:.4f}"
    row["cv_pct"] = f"{cv_pct:.1f}%"
    rows.append(row)

df_table = pd.DataFrame(rows)

# Print in a readable format
print("τ Threshold Robustness: Transfer Efficiency")
print("=" * 100)
print(f"{'Model':<14} ", end="")
for v in TAU_ORDER:
    lbl = TAU_LABELS[v]
    print(
        f"{lbl + ' (τ)':>10} {lbl + ' edges':>12} {lbl + ' size':>10} {lbl + ' TE':>8}  ",
        end="",
    )
print(f"{'CV%':>6}")
print("-" * 100)
for _, row in df_table.iterrows():
    print(f"{row['model']:<14} ", end="")
    for v in TAU_ORDER:
        print(
            f"{row[f'{v}_tau']:>10} {str(row[f'{v}_edges']):>12} {row[f'{v}_size']:>10} {row[f'{v}_te']:>8}  ",
            end="",
        )
    print(f"{row['cv_pct']:>6}")

df_table.to_csv(ANALYSIS_DIR / "04_threshold_robustness_table.csv", index=False)
print(f"\nSaved: {ANALYSIS_DIR / '04_threshold_robustness_table.csv'}")

# Also save full per-band results for reference
te.to_csv(ANALYSIS_DIR / "04_transfer_efficiency_per_tau.csv", index=False)
print(f"Saved: {ANALYSIS_DIR / '04_transfer_efficiency_per_tau.csv'}")

τ Threshold Robustness: Transfer Efficiency
Model           τ_low (τ)  τ_low edges τ_low size τ_low TE      τ* (τ)     τ* edges    τ* size    τ* TE  τ_high (τ) τ_high edges τ_high size τ_high TE     CV%
----------------------------------------------------------------------------------------------------
pythia-70m       6.31e-04          590      44.6%   0.7702    1.58e-03          436      32.9%   0.7545    3.98e-03          295      22.3%   0.7311    2.1%
pythia-160m      2.51e-04         2138      18.6%   0.9964    6.31e-04         1396      12.2%   0.9928    1.58e-03          904       7.9%   0.9645    1.4%
pythia-410m      1.00e-04         5949       7.4%   0.9932    2.51e-04         3444       4.3%   0.9838    6.31e-04         2073       2.6%   0.9679    1.1%
pythia-1b        6.31e-04         1425      14.2%   0.9669    1.58e-03          939       9.4%   0.9538    3.98e-03          551       5.5%   0.9189    2.1%
pythia-1.4b      2.51e-04         3770       4.7%   0.9127    6.31e-

## 4. Visualization

Two panels:
1. Transfer efficiency vs. circuit size (size_fraction): one line per model, 3 points (τ_low, τ*, τ_high)
2. Per-band accuracy heatmap at each τ variant (for each model)

In [4]:
# VIZ 04.01: Transfer efficiency vs circuit size
fig, ax = plt.subplots(figsize=(8, 6))

for model in MODELS:
    msub = te[te["model"] == model].sort_values("size_fraction")
    color = MODEL_COLORS[model]

    ax.plot(
        msub["size_fraction"],
        msub["transfer_eff"],
        "-",
        color=color,
        linewidth=2,
        alpha=0.8,
    )

    for _, r in msub.iterrows():
        v = r["tau_variant"]
        ax.scatter(
            r["size_fraction"],
            r["transfer_eff"],
            color=color,
            marker=TAU_MARKERS[v],
            s=80,
            zorder=5,
        )

    # Label model at the τ* point
    star_row = msub[msub["tau_variant"] == "star"]
    if len(star_row):
        ax.annotate(
            model.replace("pythia-", ""),
            (star_row["size_fraction"].values[0], star_row["transfer_eff"].values[0]),
            textcoords="offset points",
            xytext=(8, 4),
            fontsize=9,
            color=color,
            fontweight="bold",
        )

# Legend for markers
from matplotlib.lines import Line2D

legend_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="gray",
        linestyle="None",
        markersize=8,
        label="τ_low (more edges)",
    ),
    Line2D(
        [0],
        [0],
        marker="s",
        color="gray",
        linestyle="None",
        markersize=8,
        label="τ* (selected)",
    ),
    Line2D(
        [0],
        [0],
        marker="^",
        color="gray",
        linestyle="None",
        markersize=8,
        label="τ_high (fewer edges)",
    ),
]
ax.legend(handles=legend_handles, loc="lower right", fontsize=9)
ax.set_xlabel("Circuit size (fraction of total edges)", fontsize=12)
ax.set_ylabel("Transfer efficiency (cross / same-band acc)", fontsize=12)
ax.set_title(
    "Transfer Efficiency is Stable Across τ Variants\n"
    "(each line = one model, 3 points = τ_low / τ* / τ_high)",
    fontsize=12,
)
ax.set_xscale("log")

fig.tight_layout()
save_figure(fig, "viz_04_01_transfer_eff_vs_size.png")

Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/threshold_robustness/viz/viz_04_01_transfer_eff_vs_size.png


In [5]:
# VIZ 04.02: Per-band accuracy at each tau variant: heatmap per model
fig, axes = plt.subplots(1, len(MODELS), figsize=(4.5 * len(MODELS), 4.5))

BAND_LABELS = ["low", "medium", "high", "very high", "control"]

for ax, model in zip(axes, MODELS):
    msub = df[df["model"] == model].copy()
    msub["tau_variant"] = pd.Categorical(
        msub["tau_variant"], categories=TAU_ORDER, ordered=True
    )
    pivot = msub.pivot(
        index="tau_variant", columns="test_band", values="circuit_accuracy"
    )
    pivot = pivot.reindex(index=TAU_ORDER, columns=BANDS)
    pivot.index = [TAU_LABELS[v] for v in TAU_ORDER]
    pivot.columns = BAND_LABELS

    sns.heatmap(
        pivot,
        annot=True,
        fmt=".3f",
        cmap="YlGn",
        vmin=0,
        vmax=1,
        ax=ax,
        square=True,
        linewidths=0,
        linecolor="none",
    )
    ax.set_title(model, fontsize=11, fontweight="bold")
    ax.set_xlabel("Test band")
    ax.set_ylabel("")

axes[0].set_ylabel("τ variant")
fig.suptitle(
    "Circuit Accuracy per (τ Variant, Test Band)\n"
    "(rows = threshold variant, columns = test band)",
    fontsize=13,
    fontweight="bold",
)
fig.tight_layout()
save_figure(fig, "viz_04_02_accuracy_heatmap.png")

Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/threshold_robustness/viz/viz_04_02_accuracy_heatmap.png


## 5. Summary

In [6]:
print("=" * 70)
print("PHASE 5: D2: τ THRESHOLD ROBUSTNESS")
print("=" * 70)

print("\n--- Transfer Efficiency per Model x τ Variant ---")
print(f"{'Model':<14} {'τ_low TE':>10} {'τ* TE':>10} {'τ_high TE':>10} {'CV%':>8}")
print("-" * 54)
all_cv = []
for model in MODELS:
    msub = te[te["model"] == model].sort_values("tau_variant")
    te_vals = dict(zip(msub["tau_variant"], msub["transfer_eff"]))
    vals_arr = list(te_vals.values())
    cv_pct = (
        np.std(vals_arr) / np.mean(vals_arr) * 100
        if np.mean(vals_arr) > 0
        else float("nan")
    )
    all_cv.append(cv_pct)
    print(
        f"{model:<14} {te_vals.get('low', float('nan')):>10.4f} "
        f"{te_vals.get('star', float('nan')):>10.4f} "
        f"{te_vals.get('high', float('nan')):>10.4f} "
        f"{cv_pct:>7.1f}%"
    )

mean_cv = np.mean(all_cv)
print(f"\nMean CV across models: {mean_cv:.1f}%")

print("\n--- Circuit Size per τ Variant ---")
print(f"{'Model':<14} {'τ_low':>8} {'τ* size':>8} {'τ_high':>8}")
print("-" * 44)
for model in MODELS:
    msub = te[te["model"] == model].sort_values("tau_variant")
    sizes = dict(zip(msub["tau_variant"], msub["size_fraction"]))
    print(
        f"{model:<14} {sizes.get('low', float('nan')):>8.1%} "
        f"{sizes.get('star', float('nan')):>8.1%} "
        f"{sizes.get('high', float('nan')):>8.1%}"
    )

print("\n--- Key Finding ---")
if mean_cv < 5:
    print(f"Transfer efficiency is ROBUST to τ choice (mean CV = {mean_cv:.1f}% < 5%)")
    print("The τ* selection does not drive the transfer efficiency result.")
else:
    print(
        f"Transfer efficiency shows some sensitivity to τ (mean CV = {mean_cv:.1f}% >= 5%)"
    )
    print("See heatmap for per-model breakdown.")

print("\n--- Output Files ---")
for f in sorted(ANALYSIS_DIR.glob("04_*.csv")):
    print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")
for f in sorted(VIZ_DIR.glob("viz_04_*.png")):
    print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")
print("\nDone.")

PHASE 5: D2: τ THRESHOLD ROBUSTNESS

--- Transfer Efficiency per Model x τ Variant ---
Model            τ_low TE      τ* TE  τ_high TE      CV%
------------------------------------------------------
pythia-70m         0.7702     0.7545     0.7311     2.1%
pythia-160m        0.9964     0.9928     0.9645     1.4%
pythia-410m        0.9932     0.9838     0.9679     1.1%
pythia-1b          0.9669     0.9538     0.9189     2.1%
pythia-1.4b        0.9127     0.9348     0.8797     2.5%

Mean CV across models: 1.9%

--- Circuit Size per τ Variant ---
Model             τ_low  τ* size   τ_high
--------------------------------------------
pythia-70m        44.6%    32.9%    22.3%
pythia-160m       18.6%    12.2%     7.9%
pythia-410m        7.4%     4.3%     2.6%
pythia-1b         14.2%     9.4%     5.5%
pythia-1.4b        4.7%     2.6%     1.5%

--- Key Finding ---
Transfer efficiency is ROBUST to τ choice (mean CV = 1.9% < 5%)
The τ* selection does not drive the transfer efficiency result.

--- 